In [ ]:
import pandas as pd
import json
import argparse
import os
import sys
import requests as rq
import numpy as np
import matplotlib.pyplot as plt
import getpass
import pprint
from IPython.display import display, Markdown, Latex

## Get completeness stats from a running CanDIG deployment

You need to grab a token and enter it below, this notebook is configured to grab data from the UHN prod deployment.

In [ ]:
candig_url = input("Enter your CanDIG deployment URL:")
token = getpass.getpass('Enter your token:')


In [ ]:
def grab_data(token, url):
    body = {
        "method": "GET",
        "path": "discovery/programs",
        "payload": {},
        "service": "query"
    }
    headers = {"Authorization": f"Bearer {token}",
               "Content-Type": "application/json; charset=utf-8",
               "federation": "true"
               }
    response = rq.post(f"{url}/federation/v1/fanout", headers=headers, json=body)
    return response.json()

completeness_data = grab_data(token, candig_url)
# pprint.pprint(completeness_data)

## Program level completeness statistics per site by schema
The following graphs for each program at each site, which schemas are completely missing. If marked with a `0` there is not data available for that schema for that program

In [ ]:
for site in completeness_data:
    display(Markdown(f"## Node: {site['location']['name']}"))
    schemas = ["donors",
        "primary_diagnoses",
        "treatments",
        "radiations",
        "surgeries",
        "systemic_therapies",
        "specimens",
        "sample_registrations",
        "followups",
        "biomarkers"]
    summary_dict = {
        "program_id": [],
        "donors": [],
        "primary_diagnoses": [],
        "treatments": [],
        "radiations": [],
        "surgeries": [],
        "systemic_therapies": [],
        "specimens": [],
        "sample_registrations": [],
        "followups": [],
        "biomarkers": []
    }
    for program in site['results']['programs']:
        summary_dict["program_id"].append(program["program_id"])
        for schema in schemas:
            if schema in program['metadata']['schemas_not_used']:
                summary_dict[schema].append("0")
            else:
                summary_dict[schema].append("")
        #display(Markdown(f"#### Program: {program['program_id']}"))
        #display(Markdown(f"Schemas not used: {', '.join(program['metadata']['schemas_not_used'])}"))
        #site_schemas_not_used.extend(program['metadata']['schemas_not_used'])
        #formatted = ["- " + x for x in program['metadata']['schemas_not_used']]
        #display(Markdown(", ".join(program['metadata']['schemas_not_used'])))
    display(pd.DataFrame(summary_dict))

## Field level completeness statistics per site

The following graphs show overall completeness for each required field at each site in both percentage and absolute numbers.

In [ ]:
def get_stats(schema, site_stats, site_name):
    try:
        fields = list(site_stats['results']['site']['required_but_missing'][schema].keys())
        pcts = []
        totals = []
        for field, stats in site_stats['results']['site']['required_but_missing'][schema].items():
            pcts.append(round(((stats['total'] - stats['missing'])/stats['total'])*100))
            totals.append(stats['total'] - stats['missing'])
    except KeyError as e:
        print(f"{site_name} has no data for {e} schema")
        fields = []
        pcts = []
        totals = []
    # print(f"fields: {fields}, pcts: {pcts}, totals: {totals}")
    return fields, pcts, totals

In [ ]:
def plot_by_schema_per_site(schema_name, completeness_stats):
    site_names = [x['location']['name'] for x in completeness_stats]
    site_stats = {x:{} for x in site_names}

    for site in completeness_stats:
        # pprint.pprint(site)
        site_name = site['location']['name']
        fields, pct_complete, total_complete = get_stats(schema_name, site, site_name)
        site_stats[site_name] = {"fields": fields,
                                 "pct_complete": pct_complete,
                                 "totals": total_complete}
    
    if schema_name != "biomarkers":
        x = np.arange(len(fields))  # the label locations
        width = 0.25  # the width of the bars
        multiplier = 0
        
        fig, ax = plt.subplots()
        
        site_colours = ['#905ba7', '#ed1b2f', '#0f6cb7']
        pct_completeness = {'BCGSC': site_stats['BCGSC']['pct_complete'],
                           'MOH-Q': site_stats['MOH-Q']['pct_complete'],
                           'UHN': site_stats['UHN']['pct_complete']}
        
        for index, (site, completeness) in enumerate(pct_completeness.items()):
            offset = width * multiplier
            rects = ax.barh(x + offset, completeness, width, label=site, color=site_colours[index%len(site_colours)])
            ax.bar_label(rects, padding=4)
            multiplier += 1
            
        # Add some text for labels, title and custom x-axis tick labels, etc.
        ax.set_xlabel('completeness (%)')
        ax.set_title(f'{schema_name} completeness by site')
        ax.set_yticks(x + width, fields)
        ax.legend(loc='lower left')
        ax.set_xlim(0, 110)
            
        plt.show()
    
        total_completeness = {'BCGSC': site_stats['BCGSC']['totals'],
                           'MOH-Q': site_stats['MOH-Q']['totals'],
                           'UHN': site_stats['UHN']['totals']}

        x = np.arange(len(fields))  # the label locations
        width = 0.25  # the width of the bars
        multiplier = 0
        
        fig, ax = plt.subplots()
    
        for index, (site, completeness) in enumerate(total_completeness.items()):
            offset = width * multiplier
            rects = ax.barh(x + offset, completeness, width, label=site, color=site_colours[index%len(site_colours)])
            ax.bar_label(rects, padding=4)
            multiplier += 1
            
        # Add some text for labels, title and custom x-axis tick labels, etc.
        ax.set_xlabel('completeness (n)')
        ax.set_title(f'{schema_name} completeness by site')
        ax.set_yticks(x + width, fields)
        ax.legend(loc='lower left')
        

In [ ]:
schemas_list = ["donors",
        "primary_diagnoses",
        "treatments",
        "radiations",
        "surgeries",
        "systemic_therapies",
        "specimens",
        "sample_registrations",
        "followups",
        "biomarkers"]

In [ ]:
for schema in schemas_list:
    plot_by_schema_per_site(schema, completeness_data)

## Field level completeness stats by program by schema

The following graphs show field level completeness for required fields in each schema at each site.


In [ ]:
def plot_by_schema_per_program(schema_name, program_completeness_stats):
    percents = []
    fields = list(program_completeness_stats['metadata']['required_but_missing'][schema_name].keys())
    missing = [program_completeness_stats['metadata']['required_but_missing'][schema_name][x]['missing'] for x in fields]
    total = [program_completeness_stats['metadata']['required_but_missing'][schema_name][x]['total'] for x in fields]
    percents = [((b-a)/b)*100 for a,b in zip(missing,total)]
    fig, ax = plt.subplots()

    ax.barh(fields, percents)

    ax.set_xlabel('Percent complete')
    ax.set_title(f'Completeness for required {schema_name} fields for {program_completeness_stats['program_id']}')

    plt.show()

### UHN Programs

In [ ]:
for program in completeness_data[2]['results']['programs']:
    display(Markdown(f'#### {program['program_id']} completeness per schema'))
    display(Markdown(f'{program['program_id']} does not have data for {", ".join(program['metadata']['schemas_not_used'])}'))
    
    for schema in program['metadata']['schemas_used']:
        if schema == "exposures":
            continue
        plot_by_schema_per_program(schema, program)

### MOH-Q Programs

In [ ]:
for program in completeness_data[1]['results']['programs']:
    display(Markdown(f'#### {program['program_id']} completeness per schema'))
    display(Markdown(f'{program['program_id']} does not have data for {", ".join(program['metadata']['schemas_not_used'])}'))
    
    for schema in program['metadata']['schemas_used']:
        if schema == "exposures":
            continue
        plot_by_schema_per_program(schema, program)

### BCGSC Programs

In [ ]:
for program in completeness_data[0]['results']['programs']:
    display(Markdown(f'#### {program['program_id']} completeness per schema'))
    display(Markdown(f'{program['program_id']} does not have data for {", ".join(program['metadata']['schemas_not_used'])}'))
    
    for schema in program['metadata']['schemas_used']:
        if schema == "exposures":
            continue
        plot_by_schema_per_program(schema, program)